# SQL — Schema Design: Star Schema, Snowflake, SCD Type 2
---

<div style="padding:15px;border-left:8px solid #a18cd1;background:#f5f0ff;border-radius:4px;">
<strong>Core Insight:</strong> Schema design separates data engineers from analysts.
Star = denormalized for speed. Snowflake = normalized for consistency.
SCD Type 2 = the standard for tracking historical changes.
Get these three patterns wrong and your data warehouse is unusable.
</div>

### The Citi Context
Citi's capacity analytics warehouse used a star schema. When a server moved teams,
a new SCD Type 2 row was added — preserving who owned the server at incident time.
This changed root cause attribution for 12% of incidents.

## 🧠 Star vs Snowflake: The Decision

| Schema | Structure | When to use |
|--------|-----------|-------------|
| **Star** | Fact table + flat dimension tables (denormalized) | BI tools, fast queries, analysts write SQL |
| **Snowflake** | Fact table + normalized dimension tables (sub-dimensions) | Data consistency > query speed, ETL pipelines |
| **One Big Table** | Single flat table with all columns | Small datasets, ad-hoc analysis only |

### The Star Schema Pattern
```
fact_monitoring (event_id FK→servers FK→dates FK→metrics, value)
       │
       ├── dim_server (server_id, server_name, env, team, is_active)
       ├── dim_date   (date_id, date, year, quarter, month, day_of_week)
       └── dim_metric (metric_id, metric_type, unit, description)
```

**Why star queries are fast:**
- No joins between dimension tables — only fact → dim
- Dimension tables fit in memory (6,000 servers vs 500M fact rows)
- Query planner knows dimension tables are small → efficient hash joins

In [ ]:
-- Star Schema DDL — Citi capacity analytics warehouse

-- Dimension: Servers (SCD Type 2 — historical tracking)
CREATE TABLE dim_server (
    server_key      SERIAL PRIMARY KEY,          -- surrogate key (never changes)
    server_id       VARCHAR(20) NOT NULL,         -- natural key (SRV-1042)
    server_name     VARCHAR(100) NOT NULL,
    environment     VARCHAR(20) NOT NULL,          -- dev/staging/prod
    team            VARCHAR(50) NOT NULL,
    application     VARCHAR(100),
    data_center     VARCHAR(50),
    is_active       BOOLEAN NOT NULL DEFAULT TRUE,
    -- SCD Type 2 tracking columns
    effective_date  DATE NOT NULL,               -- when this version became active
    expiry_date     DATE,                         -- NULL = current record
    is_current      BOOLEAN NOT NULL DEFAULT TRUE,
    -- Audit columns
    created_at      TIMESTAMP DEFAULT NOW(),
    updated_at      TIMESTAMP DEFAULT NOW()
);

-- Dimension: Dates (pre-populated calendar table)
CREATE TABLE dim_date (
    date_key        INTEGER PRIMARY KEY,          -- YYYYMMDD as integer (fast join)
    full_date       DATE NOT NULL,
    year            INTEGER NOT NULL,
    quarter         INTEGER NOT NULL,             -- 1-4
    month           INTEGER NOT NULL,             -- 1-12
    month_name      VARCHAR(10) NOT NULL,
    week_of_year    INTEGER NOT NULL,
    day_of_week     INTEGER NOT NULL,             -- 1=Monday, 7=Sunday
    is_weekend      BOOLEAN NOT NULL,
    is_holiday      BOOLEAN NOT NULL DEFAULT FALSE
);

-- Dimension: Metrics
CREATE TABLE dim_metric (
    metric_id       SERIAL PRIMARY KEY,
    metric_type     VARCHAR(50) NOT NULL,         -- cpu_pct, memory_pct, latency_ms
    unit            VARCHAR(20) NOT NULL,
    description     TEXT
);

-- Fact table: Monitoring events
CREATE TABLE fact_monitoring (
    event_id        BIGSERIAL PRIMARY KEY,
    server_key      INTEGER NOT NULL REFERENCES dim_server(server_key),
    date_key        INTEGER NOT NULL REFERENCES dim_date(date_key),
    metric_id       INTEGER NOT NULL REFERENCES dim_metric(metric_id),
    collected_at    TIMESTAMP NOT NULL,
    metric_value    NUMERIC(8,3) NOT NULL,
    -- Partitioned by month for performance
    CONSTRAINT fact_monitoring_collected_at CHECK (collected_at IS NOT NULL)
) PARTITION BY RANGE (collected_at);

-- Monthly partitions
CREATE TABLE fact_monitoring_2024_01 PARTITION OF fact_monitoring
    FOR VALUES FROM ('2024-01-01') TO ('2024-02-01');

-- Indexes
CREATE INDEX idx_fact_server_date ON fact_monitoring(server_key, date_key);
CREATE INDEX idx_fact_metric_date ON fact_monitoring(metric_id, date_key);

In [ ]:
-- SCD Type 2: Handling server team changes
-- When SRV-1042 moves from Team A to Team B on 2024-03-15,
-- we CLOSE the old record and INSERT a new one. History is preserved.

-- Current state: SRV-1042 belongs to Team A (effective 2023-01-01)
SELECT server_id, team, effective_date, expiry_date, is_current
FROM dim_server WHERE server_id = 'SRV-1042';
-- Returns: SRV-1042 | Team A | 2023-01-01 | NULL | TRUE

-- ── SCD Type 2 Update Procedure ──────────────────────────────────────────────
-- Step 1: Close the current record
UPDATE dim_server
SET expiry_date = '2024-03-14',
    is_current  = FALSE,
    updated_at  = NOW()
WHERE server_id  = 'SRV-1042'
  AND is_current = TRUE;

-- Step 2: Insert new version
INSERT INTO dim_server (server_id, server_name, environment, team, effective_date, expiry_date, is_current)
VALUES ('SRV-1042', 'SRV-1042', 'prod', 'Team B', '2024-03-15', NULL, TRUE);

-- ── Now you can query historical ownership ─────────────────────────────────
-- "Which team owned SRV-1042 during the January 2024 incident?"
SELECT s.server_id, s.team, s.effective_date, s.expiry_date
FROM dim_server s
WHERE s.server_id = 'SRV-1042'
  AND s.effective_date <= '2024-01-15'
  AND (s.expiry_date IS NULL OR s.expiry_date >= '2024-01-15');
-- Returns: SRV-1042 | Team A | 2023-01-01 | 2024-03-14

-- Critical insight: without SCD Type 2, a server that moved teams would ALWAYS show
-- the CURRENT team — making historical incident attribution wrong.
PRINT("SCD Type 2 preserves ownership at incident time — critical for post-mortems")

## 📊 Snowflake Schema: When Normalization Wins

```
fact_monitoring
    └── dim_server (server_key, server_id, location_key, team_key)
            ├── dim_location (location_key, data_center, city, country)
            └── dim_team     (team_key, team_name, department, cost_center)
```

**When to use Snowflake:**
- Location and team data changes frequently and independently
- You need a single source of truth for each entity
- You're building ETL pipelines (not analyst-facing BI)

**Trade-off:** Every query now needs 3-4 JOINs instead of 1-2.
For ad-hoc SQL by analysts, this creates friction.

```sql
-- Star query (simpler for analysts):
SELECT s.team, AVG(f.metric_value)
FROM fact_monitoring f
JOIN dim_server s ON f.server_key = s.server_key
GROUP BY s.team;

-- Snowflake query (more joins, more maintenance):
SELECT t.team_name, AVG(f.metric_value)
FROM fact_monitoring f
JOIN dim_server s   ON f.server_key = s.server_key
JOIN dim_team t     ON s.team_key = t.team_key
GROUP BY t.team_name;
```

## 🎤 5 Interview Q&A

**Q1: What is the difference between a star schema and a snowflake schema?**
Star: fact table + flat (denormalized) dimension tables. All dimension attributes in one table.
Snowflake: dimensions are normalized — hierarchies split into separate tables (dim_server → dim_location → dim_country).
Star is faster for queries (fewer joins), snowflake is more consistent for updates (each entity has one record).
Use star for BI/analytics. Use snowflake when dimension data changes frequently and independently.

---

**Q2: What is SCD Type 2 and when do you need it?**
SCD = Slowly Changing Dimension. Type 2 preserves history by adding new rows.
When an attribute changes (server moves teams, customer changes address), you:
(1) Close the current row by setting `expiry_date` and `is_current = FALSE`
(2) Insert a new row with the new values, `effective_date = today`, `is_current = TRUE`
You need it when you must answer questions like "which team owned this server during the incident?"
— questions that require historical accuracy, not just current state.

---

**Q3: What is a surrogate key and why use it instead of a natural key?**
Natural key = real-world identifier (server_id = 'SRV-1042').
Surrogate key = system-generated integer (server_key = 4521).
Surrogate keys are used as foreign keys in fact tables because:
(1) They never change — even if the natural key changes (server renamed)
(2) They're smaller (integer vs varchar) — faster joins
(3) They support SCD Type 2 — multiple rows for the same server_id get different server_keys

---

**Q4: What is a date dimension table and why not just use the date column?**
A dim_date table pre-computes date attributes: year, quarter, month, day_of_week, is_weekend, is_holiday.
Without it, every query involving "group by quarter" must compute `DATE_TRUNC('quarter', collected_at)`.
With a date dimension: `JOIN dim_date ON date_key = collected_at::DATE` → filter `WHERE quarter = 1`.
Also: you can mark company holidays, fiscal calendar periods, and other business rules once in dim_date
instead of in every query.

---

**Q5: What is a fact table and what should (and shouldn't) go in it?**
Fact table: stores events/measurements with foreign keys to dimensions and numeric measures.
✅ IN fact table: event timestamp, surrogate keys to dimensions, numeric measures (cpu_pct, latency_ms, count)
❌ NOT in fact table: descriptive attributes (server_name, team) — those go in dimensions.
This separation enables the star schema's core property: add new dimensions without changing the fact table.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Fact Table** | Central table storing events/measurements with numeric measures and foreign keys to dimensions |
| **Dimension Table** | Reference table storing descriptive attributes of entities (servers, dates, metrics) |
| **Star Schema** | Fact table + flat (denormalized) dimension tables — optimized for query speed |
| **Snowflake Schema** | Fact table + normalized dimension hierarchy — optimized for consistency |
| **SCD Type 1** | Overwrite the old value — no history. Simple but lossy. |
| **SCD Type 2** | Add new row with new values, mark old row expired — preserves full history |
| **SCD Type 3** | Add a "previous value" column — limited history (only 1 change tracked) |
| **Surrogate Key** | System-generated integer primary key — never changes, used for foreign keys |
| **Natural Key** | Real-world identifier (server_id, customer_email) — may change |
| **Grain** | The level of detail in a fact table — one row per server per metric per minute |
| **Slowly Changing Dimension** | A dimension whose attributes change infrequently over time |
| **Conformed Dimension** | A dimension shared across multiple fact tables — ensures consistent joins |

## 💼 The Citi Narrative

**Context:** Citi capacity analytics — needed a data warehouse to support both real-time
alerting and quarterly trend reporting across 6,000 servers.

**The Schema Decision:**
- Star schema chosen over snowflake: analysts write SQL directly, and query simplicity wins
- SCD Type 2 on dim_server: servers moved between teams during organizational restructures (3-4 times/year)
- Partitioned fact table: 500M rows by year-end, needed partition pruning for monthly reports

**The SCD Type 2 Impact:**
Before SCD Type 2: incident post-mortem would show "Team B owns SRV-1042" — but the incident
happened when Team A still owned it. Root cause attribution was wrong for servers that had
changed ownership.

After SCD Type 2: `WHERE effective_date <= incident_date AND (expiry_date IS NULL OR expiry_date >= incident_date)`
correctly identified Team A as the responsible team at incident time.

**Impact:** Changed root cause attribution for 12% of incidents in the first quarter after
implementing SCD Type 2. Infrastructure teams could no longer claim "that server moved to
another team before the incident" — the data now showed the truth.

**Interview Line:** *"We implemented SCD Type 2 because teams argued about who owned a server
during an incident. With the old system, server ownership was current-state only — if a server
moved teams before the post-mortem, the incident would blame the new team. SCD Type 2 let us
ask 'who owned it at 2am on January 15th?' and get the right answer."*

In [ ]:
-- Practice: Write SCD Type 2 queries

-- Table: dim_server (server_key, server_id, team, effective_date, expiry_date, is_current)
-- Table: fact_monitoring (server_key, collected_at, metric_value)

-- 1. Find the team responsible for all HIGH-CPU incidents in January 2024
-- (servers with metric_value > 90 at any point in January)
-- Use SCD Type 2 to get the team that owned the server DURING the incident

-- 2. Count servers by team as of March 15, 2024
-- (point-in-time query)

-- Solutions:
print("Query 1: Team responsible for January incidents")
print("""
SELECT s.team, COUNT(*) AS high_cpu_events
FROM fact_monitoring f
JOIN dim_server s ON f.server_key = s.server_key
WHERE f.metric_value > 90
  AND f.collected_at BETWEEN '2024-01-01' AND '2024-01-31'
  AND s.effective_date <= f.collected_at::DATE
  AND (s.expiry_date IS NULL OR s.expiry_date >= f.collected_at::DATE)
GROUP BY s.team
ORDER BY high_cpu_events DESC;
""")

print("Query 2: Server count by team on March 15, 2024")
print("""
SELECT team, COUNT(*) AS server_count
FROM dim_server
WHERE effective_date <= '2024-03-15'
  AND (expiry_date IS NULL OR expiry_date >= '2024-03-15')
GROUP BY team
ORDER BY server_count DESC;
""")

## 🎯 Summary

### The Three Schema Patterns
| Pattern | Use when | Trade-off |
|---------|----------|-----------|
| **Star** | Analysts write SQL, BI tools | Faster queries, some redundancy |
| **Snowflake** | ETL pipelines, consistency critical | More joins, cleaner updates |
| **SCD Type 2** | Historical accuracy required | More rows, more complex queries |

### Interview Confidence Checklist
- [ ] Can draw a star schema with fact + 3 dimensions
- [ ] Can explain SCD Type 2 with the 2-step update (close old, insert new)
- [ ] Can write the point-in-time join query with effective/expiry dates
- [ ] Can explain surrogate vs natural key
- [ ] Can name the Citi narrative: 12% of incidents had wrong attribution before SCD Type 2

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀